# Modified pix2pixHD — Test and Geospatial Post-processing

This notebook is designed to test trained weights using the reorganized pix2pixHD implementation. It supports the coordinate-conditioned pix2pixHD model. 

Geographic unit-sphere XYZ metadata are supplied separately from RGB images and introduced at the generator bottleneck.

Set `NAME` to the trained experiment. The generator configuration (`global` or `local`), normalization, and `which_epoch` must match the checkpoint.

Coordinate metadata is required. By default, the model reads `DATAROOT/geo_coordinates.csv`; `COORD_CSV` can specify another path. Use `generate_coordinate_csv.ipynb` to prepare the file.

The generated `geo_coordinates.csv` file should then be kept in the dataset root directory and used together with the corresponding `train_A`, `train_B`, `test_A`, and `test_B` folders.

## Dataset structure

`DATAROOT` must point to the root directory of the dataset. The recommended pix2pixHD dataset structure is:

```text
DATAROOT/
├── train_A/
├── train_B/
├── test_A/
└── test_B/
└── geo_coordinates.csv


In [ ]:
from pathlib import Path
import sys
import subprocess

# Open the notebook from the project root or the notebooks/ directory.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "train.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "train.py").is_file(), "Set PROJECT_ROOT to the project directory."
print("Python:", sys.executable)
print("Project:", PROJECT_ROOT)


In [ ]:
# Dataset and checkpoint settings.

DATAROOT = PROJECT_ROOT / "datasets"

NAME = "Model_V2_dataset_global_D3_allloss_aug_batch_coord_bottleneck"

COORD_CSV = ""  # Empty: DATAROOT/geo_coordinates.csv
GPU_IDS = "0"  # Use "-1" for CPU (a GPU is recommended for training)
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints"


In [ ]:
settings = {
    "name": NAME,
    "dataroot": str(DATAROOT),
    "checkpoints_dir": str(CHECKPOINTS_DIR),

    "label_nc": 0,
    "input_nc": 3,
    "output_nc": 3,

    "gpu_ids": GPU_IDS,

    "batchSize": 1,
    "loadSize": 256,
    "fineSize": 256,
    "resize_or_crop": "none",

    "netG": "global",
    "n_downsample_global": 4,
    "n_blocks_global": 9,
    "ngf": 64,
    "norm": "batch",

    "nThreads": 0,

    "which_epoch": "200",
    "phase": "test",
    "how_many": 999999,

    "results_dir": str(PROJECT_ROOT / "results"),
}

if COORD_CSV:
    settings["coord_csv"] = COORD_CSV

flags = [
    "no_instance",
    "serial_batches",
    "no_flip",
]



In [ ]:
assert DATAROOT.is_dir(), f"Dataset directory not found: {DATAROOT}"
csv_path = Path(COORD_CSV) if COORD_CSV else DATAROOT / "geo_coordinates.csv"
assert csv_path.is_file(), f"Coordinate CSV not found: {csv_path}"
assert (DATAROOT / "test_A").is_dir()
checkpoint = CHECKPOINTS_DIR / NAME / f"{settings['which_epoch']}_net_G.pth"
assert checkpoint.is_file(), f"Checkpoint not found: {checkpoint}"
command = [sys.executable, "-u", str(PROJECT_ROOT / "test.py")]
for key, value in settings.items():
    command.extend(["--" + key, str(value)])
command.extend("--" + flag for flag in flags)
print(subprocess.list2cmdline(command))


In [ ]:
# This cell starts the process. 
process = subprocess.Popen(command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, errors="replace", bufsize=1)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    returncode = process.wait()
    if returncode:
        raise RuntimeError(f"Process exited with error code: {returncode}")
except KeyboardInterrupt:
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
    raise


## Georeferencing

Run this section after test inference has finished successfully. For existing results, you can run the setup/settings cells above and skip the inference cells.

This step requires `osgeo.gdal` to be available in the active notebook kernel. In this workflow, GDAL was not installed in the pix2pixHD environment in order to avoid possible dependency conflicts. Therefore, the georeferencing and mosaicking steps can be executed in a separate GDAL-enabled Python environment.

If a different environment or Jupyter kernel is used for these steps, run the first three setup cells at the beginning of the notebook again so that the required project paths, dataset paths, and configuration variables are defined in the new environment. It copies the CRS and affine geotransform from matching reference GeoTIFFs; it does not perform image registration. The predictions and reference tiles must cover the same ground footprint.

Georeferencing uses the input OSM GeoTIFFs in `test_A` as spatial references. Their CRS and affine geotransform are copied to the generated outputs. Reference satellite images and training data are not required.


In [ ]:
# Import the post-processing helpers from this project.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from util.geospatial import georeference_results, merge_georeferenced_tiles

# Derive paths from the same settings used for inference.
RESULT_DIR = (Path(settings["results_dir"]) / settings["name"] /
              f"{settings['phase']}_{settings['which_epoch']}")

dataset_dir = Path(settings["dataroot"])
test_b_dir = dataset_dir / "test_B"
has_test_b = test_b_dir.is_dir() and any(p.is_file() and p.suffix.lower() in (".tif", ".tiff") for p in test_b_dir.rglob("*"))
REFERENCE_DIR = test_b_dir if has_test_b else dataset_dir / "test_A"

PREDICT_DIR = RESULT_DIR / "images"
GEOREF_OUTPUT_DIR = RESULT_DIR / "geotiff"

GEOREF_KIND = "all"  # "all", "input_label", or "synthesized_image"
RESIZE_TO_REFERENCE = True  # Preserve the supplied notebook's default.
OVERWRITE_GEOREF = False  # Set True only to replace existing GeoTIFF outputs.
print("Reference directory:", REFERENCE_DIR)
print("Prediction directory:", PREDICT_DIR)
print("GeoTIFF output directory:", GEOREF_OUTPUT_DIR)


In [ ]:
# Before each georeferencing run, the previous file list is cleared to avoid reusing old results. 
# If georeferencing succeeds, only the newly created files are passed to the merge/mosaic step.
georeferenced_files = None
georeferenced_files = georeference_results(
    dataset_dir=REFERENCE_DIR,
    predict_dir=PREDICT_DIR,
    output_dir=GEOREF_OUTPUT_DIR,
    only=GEOREF_KIND,
    resize_to_ref=RESIZE_TO_REFERENCE,
    overwrite=OVERWRITE_GEOREF,
)


## Merge / mosaic

Run this section after georeferencing has been completed successfully. The georeferenced tiles are grouped by their original image prefix and merged separately for the input labels and synthesized images. Nearest-neighbor resampling is used for input labels, while bilinear resampling is used for synthesized RGB images.


In [ ]:
# Keep the original output layout within the geotiff directory.
MERGED_OUTPUT_DIR = GEOREF_OUTPUT_DIR
INPUT_RESAMPLE = "near"
SYNTHESIZED_RESAMPLE = "bilinear"
OVERWRITE_MERGED = False

assert georeferenced_files, "Run the georeferencing section successfully before merging."
merged_files = merge_georeferenced_tiles(
    tile_paths=georeferenced_files,
    merged_output_dir=MERGED_OUTPUT_DIR,
    input_resample=INPUT_RESAMPLE,
    synthesized_resample=SYNTHESIZED_RESAMPLE,
    overwrite=OVERWRITE_MERGED,
)
print("Mosaics created:", len(merged_files))
for path in merged_files:
    print(path)
